# GCC-4K committed EVALUATE
**THIS NOTEBOOK IS DESIGNED FOR: KAGGLE → SAVE VERSION → SAVE & RUN ALL**
Attach ONLY the successful committed TRAIN Version output. It already contains the canonical recovery tree; do not attach recovery input separately. This notebook never trains.

In [ ]:
# 1 — isolate GPU 0 before torch import
import os
os.environ['CUDA_VISIBLE_DEVICES']='0'

In [ ]:
# 2 — environment audit
import platform,subprocess,sys
subprocess.run(['nvidia-smi'],check=True)
import torch
assert torch.cuda.is_available(),'KAGGLE_CUDA_GPU_REQUIRED';assert torch.cuda.device_count()==1,'GCC4K_REQUIRES_EXACTLY_ONE_VISIBLE_GPU'
props=torch.cuda.get_device_properties(0);print({'execution_surface':'KAGGLE','gpu':props.name,'vram_bytes':props.total_memory,'cuda':torch.version.cuda,'torch':torch.__version__,'python':sys.version,'platform':platform.platform()})

In [ ]:
# 3 — pinned dependencies
import importlib.util
if importlib.util.find_spec('torchao'):subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.51.3','peft==0.20.0','accelerate','safetensors','psutil'],check=True)
os.environ['WANDB_DISABLED']='true';os.environ['HF_HUB_DISABLE_TELEMETRY']='1'

In [ ]:
# 4 — discover and verify exploded-or-ZIP canonical recovery input
import hashlib,json,shutil,zipfile
from pathlib import Path,PurePosixPath
INPUT=Path('/kaggle/input');WORK=Path('/kaggle/working/gcc4k');EXPECTED_ZIP='ab13d4b3bc7c6d19b8f83f0fc5e764687dde8acdc29239ff8d4782750e80205c';EXPECTED_CORPUS='sha256:bd1b19fc6733cca6622051e30ac03dc3cbac3a602997894c7b4a1733467f3619'
REQUIRED=['SHA256SUMS.txt','corpus/manifest.json','corpus/train.jsonl','corpus/validation.jsonl','corpus/qualification.jsonl','corpus/holdout.jsonl','corpus/adversarial.jsonl','scripts/train_targeted_student_v02.py','scripts/gcc4k_recovery.py','scripts/experiment.config.json','v01/adapter/adapter_model.safetensors']
roots=[p for p in INPUT.rglob('SHA256SUMS.txt') if all((p.parent/r).is_file() for r in REQUIRED)]
if len(roots)>1:raise RuntimeError('MULTIPLE_GCC4K_EXPLODED_INPUTS')
if roots:source=roots[0].parent;input_mode='EXPLODED_KAGGLE_DATASET';outer_zip_digest='NOT_APPLICABLE_KAGGLE_EXPLODED_INPUT'
else:
    archives=list(INPUT.rglob('PA-INTERPRETATION-STUDENT-v0.2-GCC4K-RECOVERY-INPUT.zip'))
    if len(archives)!=1:raise RuntimeError('GCC4K_INPUT_NOT_FOUND' if not archives else 'MULTIPLE_GCC4K_ZIP_INPUTS')
    archive=archives[0];actual=hashlib.sha256(archive.read_bytes()).hexdigest();assert actual==EXPECTED_ZIP,'GCC4K_RECOVERY_INPUT_SHA256_MISMATCH';source=Path('/kaggle/working/gcc4k-source');shutil.rmtree(source,ignore_errors=True)
    with zipfile.ZipFile(archive) as z:
        for name in z.namelist():path=PurePosixPath(name);assert not path.is_absolute() and '..' not in path.parts and '\\' not in name,'UNSAFE_ZIP_PATH'
        z.extractall(source)
    input_mode='RECOVERY_ZIP';outer_zip_digest=actual
verified=0
for line in (source/'SHA256SUMS.txt').read_text().splitlines():
    expected,relative=line.split('  ',1);path=PurePosixPath(relative);assert not path.is_absolute() and '..' not in path.parts and '\\' not in relative,'UNSAFE_GCC4K_PAYLOAD_PATH';payload=source/relative;assert payload.is_file(),relative;assert hashlib.sha256(payload.read_bytes()).hexdigest()==expected,relative;verified+=1
assert json.loads((source/'corpus/manifest.json').read_text())['aggregate_corpus_digest']==EXPECTED_CORPUS,'GCC4K_CORPUS_DIGEST_MISMATCH';print({'input_mode':input_mode,'outer_zip_digest':outer_zip_digest,'payload_hashes_verified':verified})

In [ ]:
# 5 — materialize verified canonical input
shutil.rmtree(WORK,ignore_errors=True);shutil.copytree(source,WORK);assert all((WORK/r).is_file() for r in REQUIRED)

In [ ]:
# 6 — select the TRAIN-manifest-bound adapter deterministically
run_inputs=list(INPUT.rglob('PA-INTERPRETATION-STUDENT-v0.2-TRAIN-RUN.json'))
if not run_inputs:raise RuntimeError('TRAIN_RUN_MANIFEST_NOT_FOUND')
if len(run_inputs)!=1:raise RuntimeError('MULTIPLE_TRAIN_RUN_MANIFESTS')
run_input=run_inputs[0];train_run=json.loads(run_input.read_text());expected_adapter_sha=train_run['adapter_zip_sha256']
adapter_inputs=list(INPUT.rglob('PA-INTERPRETATION-STUDENT-v0.2-ADAPTER-GCC4K.zip'));adapter_candidates=[(path,hashlib.sha256(path.read_bytes()).hexdigest()) for path in adapter_inputs];matching_adapters=[path for path,digest in adapter_candidates if digest==expected_adapter_sha]
if not matching_adapters:raise RuntimeError('PRESERVED_V02_ADAPTER_NOT_FOUND')
sibling_matches=[path for path in matching_adapters if path.parent==run_input.parent]
if len(sibling_matches)>1:raise RuntimeError('MULTIPLE_MANIFEST_SIBLING_ADAPTERS')
adapter_input=sibling_matches[0] if sibling_matches else sorted(matching_adapters,key=lambda path:path.as_posix())[0];adapter_input_sha=hashlib.sha256(adapter_input.read_bytes()).hexdigest();assert adapter_input_sha==expected_adapter_sha,'PRESERVED_ADAPTER_ZIP_HASH_MISMATCH'
adapter_handoff={'adapter_candidate_count':len(adapter_inputs),'adapter_matching_count':len(matching_adapters),'adapter_selected_path':str(adapter_input),'adapter_input_sha256':adapter_input_sha};print(adapter_handoff)

In [ ]:
# 7 — verify internal identity and materialize canonical final-adapter layout
sys.path.insert(0,str(WORK/'scripts'));from gcc4k_recovery import identity,promote_directory,valid_final_adapter
config=json.loads((WORK/'scripts/experiment.config.json').read_text());corpus=json.loads((WORK/'corpus/manifest.json').read_text());expected_identity=identity(config,corpus['aggregate_corpus_digest'])
for key,value in expected_identity.items():assert train_run.get(key)==value,f'TRAIN_ADAPTER_IDENTITY_MISMATCH:{key}'
staging=Path('/kaggle/working/adapter-input');shutil.rmtree(staging,ignore_errors=True)
with zipfile.ZipFile(adapter_input) as z:
    for name in z.namelist():path=PurePosixPath(name);assert not path.is_absolute() and '..' not in path.parts and '\\' not in name,'UNSAFE_ADAPTER_ZIP_PATH'
    z.extractall(staging)
extracted=staging/'PA-INTERPRETATION-STUDENT-v0.2-ADAPTER';assert valid_final_adapter(extracted,expected_identity),'INVALID_PRESERVED_FINAL_ADAPTER'
PERSIST_ROOT='/kaggle/working/PlannerAgent/GCC4K';os.environ['PLANNERAGENT_GCC4K_PERSIST_ROOT']=PERSIST_ROOT;candidate=Path(PERSIST_ROOT)/config['candidate_id'];candidate.mkdir(parents=True,exist_ok=True);promote_directory(extracted,candidate/'final-adapter');assert valid_final_adapter(candidate/'final-adapter',expected_identity)

In [ ]:
# 8 — exact pinned-base/API preflight
import inspect,peft,transformers
from huggingface_hub import HfApi
from transformers import Trainer,TrainingArguments
assert transformers.__version__=='4.51.3';assert peft.__version__=='0.20.0';assert 'eval_strategy' in inspect.signature(TrainingArguments.__init__).parameters
try:HfApi().model_info('Qwen/Qwen3-0.6B-Base',revision='da87bfb608c14b7cf20ba1ce41287e8de496c0cd')
except Exception as error:raise RuntimeError(f'KAGGLE_INTERNET_OR_PINNED_BASE_UNAVAILABLE: {error}') from error

In [ ]:
# 9 — EVALUATE ONLY
subprocess.run([sys.executable,str(WORK/'scripts/train_targeted_student_v02.py'),'--phase','evaluate','--persist-root',PERSIST_ROOT],check=True)

In [ ]:
# 10 — verify final GCC-4K result package
result=candidate/'result'/'PA-INTERPRETATION-STUDENT-v0.2-GCC4K.zip';result_manifest=json.loads((candidate/'result/final-result.manifest.json').read_text());assert (candidate/'result/FINAL_RESULT_COMPLETE').is_file();result_sha=hashlib.sha256(result.read_bytes()).hexdigest();assert result_sha==result_manifest['zip_sha256']
with zipfile.ZipFile(result) as z:assert z.testzip() is None,'FINAL_RESULT_ZIP_CORRUPT'

In [ ]:
# 11 — emit top-level committed evaluation outputs; final cell
top_result=Path('/kaggle/working/PA-INTERPRETATION-STUDENT-v0.2-GCC4K.zip');shutil.copy2(result,top_result);assert hashlib.sha256(top_result.read_bytes()).hexdigest()==result_sha
evaluation_run={'candidate_id':config['candidate_id'],'base_model':config['base_model'],'base_revision':config['base_revision'],'corpus_id':config['corpus_id'],'corpus_digest':EXPECTED_CORPUS,'training_config_digest':expected_identity['training_config_digest'],'adapter_artifact_digest':train_run['adapter_artifact_digest'],'adapter_input_sha256':adapter_input_sha,'result_sha256':result_sha,'evaluation_complete':True,'execution_surface':'KAGGLE','persistent_adapter_verified':True,'input_mode':input_mode}
run_path=Path('/kaggle/working/PA-INTERPRETATION-STUDENT-v0.2-EVALUATION-RUN.json');run_path.write_text(json.dumps(evaluation_run,sort_keys=True,indent=2)+'\n');assert top_result.is_file() and run_path.is_file();print('KAGGLE_COMMITTED_EVALUATION_OUTPUT_READY');print({'result_path':str(top_result),'size':top_result.stat().st_size,'sha256':result_sha,'evaluation_run_manifest':str(run_path)})